# Regularisation — Linear Regression (Ridge & Lasso)

**Problem:** when you have many features and limited data, linear regression overfits — coefficients blow up, model captures noise.

**Solution:** add a penalty on the size of the coefficients. This shrinks them — preferring simpler, more stable models.

| Method | Penalty | Effect |
|--------|---------|--------|
| Ridge (L2) | λ × Σβ² | Shrinks all coefficients toward 0 (never exactly 0) |
| Lasso (L1) | λ × Σ\|β\| | Shrinks some coefficients exactly to 0 — feature selection |

Here `λ` is the **regularisation strength** — a hyperparameter. In sklearn it's called `alpha`.

---

## Pipeline

1. Generate synthetic data with informative + noisy features
2. Plain Linear Regression — overfits, large coefficients
3. Ridge — shrinks coefficients
4. Lasso — sets some coefficients to exactly 0
5. Compare side-by-side

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

## Step 1 — Generate Data

- 100 samples
- 20 features total
- Only 5 features are actually informative (related to y)
- 15 are pure noise

This setup mimics a real situation where you have many candidate features but only a handful actually matter.

In [ ]:
X, y, true_coef = make_regression(
    n_samples=100,
    n_features=20,
    n_informative=5,        # only 5 features are useful
    noise=10,
    coef=True,              # return the true underlying coefficients
    random_state=42
)

print('Shape of X:', X.shape)
print('Number of truly informative features:', np.sum(true_coef != 0))
print('Number of noise features:', np.sum(true_coef == 0))

In [ ]:
# Standardise features — regularisation is scale-sensitive
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
print('Train:', X_train.shape, '  Test:', X_test.shape)

## Step 2 — Plain Linear Regression (No Regularisation)

This is the baseline — let's see how it handles the mix of useful and noise features.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

print(f'Train R2: {lr.score(X_train, y_train):.3f}')
print(f'Test  R2: {lr.score(X_test, y_test):.3f}')
print(f'\nLargest |coefficient|: {np.max(np.abs(lr.coef_)):.2f}')
print(f'Coefficient magnitudes (sorted):\n{np.round(sorted(np.abs(lr.coef_), reverse=True), 2)}')

**Observation:** plain LR assigns non-zero coefficients to **all 20 features** — including the noise ones — leading to potential overfitting and a model that's hard to interpret.

---

## Step 3 — Ridge (L2 Regularisation)

Ridge adds the penalty λ × Σβ². It shrinks coefficients toward zero — but never exactly to zero.

Higher `alpha` (sklearn's name for λ) → more shrinkage.

In [ ]:
ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train)

print(f'Train R2: {ridge.score(X_train, y_train):.3f}')
print(f'Test  R2: {ridge.score(X_test, y_test):.3f}')
print(f'\nLargest |coefficient|: {np.max(np.abs(ridge.coef_)):.2f}')
print(f'Number of coefficients exactly = 0: {np.sum(ridge.coef_ == 0)}')

**Observation:** all 20 coefficients are still non-zero, but they're smaller in magnitude. Ridge stabilises the model but does not perform feature selection.

---

## Step 4 — Lasso (L1 Regularisation)

Lasso adds the penalty λ × Σ|β|. The absolute value penalty can force coefficients to be **exactly zero** — Lasso effectively performs feature selection.

In [ ]:
lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)

print(f'Train R2: {lasso.score(X_train, y_train):.3f}')
print(f'Test  R2: {lasso.score(X_test, y_test):.3f}')
print(f'\nNumber of coefficients exactly = 0: {np.sum(lasso.coef_ == 0)}')
print(f'Number of non-zero coefficients   : {np.sum(lasso.coef_ != 0)}')
print(f'\nNon-zero coefficients: {np.round(lasso.coef_[lasso.coef_ != 0], 2)}')

**Observation:** Lasso has zeroed out many features — keeping only the most useful ones. This is **automatic feature selection**.

---

## Step 5 — Side-by-Side Comparison

In [ ]:
comparison = pd.DataFrame({
    'feature':   [f'X{i}' for i in range(20)],
    'true_coef': np.round(true_coef, 2),
    'linear':    np.round(lr.coef_, 2),
    'ridge':     np.round(ridge.coef_, 2),
    'lasso':     np.round(lasso.coef_, 2)
})
print(comparison)

In [ ]:
summary = pd.DataFrame({
    'Model':      ['Linear', 'Ridge', 'Lasso'],
    'Train R2':   [lr.score(X_train, y_train), ridge.score(X_train, y_train), lasso.score(X_train, y_train)],
    'Test  R2':   [lr.score(X_test,  y_test),  ridge.score(X_test,  y_test),  lasso.score(X_test,  y_test)],
    'Non-zero coefs': [np.sum(lr.coef_ != 0), np.sum(ridge.coef_ != 0), np.sum(lasso.coef_ != 0)]
})
print(summary.round(3))

## Effect of alpha

Larger alpha = stronger penalty = more shrinkage. Let's sweep alpha for Lasso and watch coefficients shrink.

In [ ]:
alphas = [0.001, 0.01, 0.1, 1.0, 5.0, 10.0]
rows = []
for a in alphas:
    m = Lasso(alpha=a, max_iter=10000).fit(X_train, y_train)
    rows.append({
        'alpha': a,
        'nonzero_coefs': np.sum(m.coef_ != 0),
        'test_R2': round(m.score(X_test, y_test), 3)
    })
print(pd.DataFrame(rows))

As alpha grows, more features get zeroed out. There's a sweet spot — too little = overfitting, too much = underfitting (model is too simple).

Use `GridSearchCV` (from the Cross-Validation notebook) to pick the best alpha.

---

## Summary

| | Ridge (L2) | Lasso (L1) |
|-|------------|------------|
| Penalty | λ Σβ² | λ Σ\|β\| |
| Shrinks toward 0? | Yes | Yes |
| Forces coefficients = 0? | No | Yes — feature selection |
| When to use | Many correlated features | Want a sparse / interpretable model |

> Always **standardise features** before applying Ridge or Lasso — the penalty is sensitive to feature scale.